# Mental Health Signal Detector — Fine-tuning v3 (Kaggle)
**Artefact School of Data — Bootcamp Data Science, Mars 2026**

## Adaptation Kaggle vs Colab

Ce notebook est adapté de `mental_bert_finetune_v3.ipynb` (version Colab).
Il utilise le GPU gratuit Kaggle (T4 × 2 ou P100) sans dépendance Google Drive.

### Datasets à ajouter dans l'onglet "Add Data" (Kaggle sidebar)

| Dataset Kaggle | Chemin dans le notebook |
|---|---|
| `bhavikjikadara/mental-health-dataset` | `/kaggle/input/mental-health-dataset/` |
| `nikhileswarkomati/suicide-ideation-dataset` | `/kaggle/input/suicide-ideation-dataset/` |

> **Note eRisk25** : données CLEF 2025 soumises à conditions d'utilisation strictes.
> Si vous y avez accès, uploadez-les dans un dataset privé Kaggle et ajustez `ERISK25_PATH`.

### Stratégie v3 — Domaine clinique strict

**Problème identifié dans v2** : mix données générales (DAIR-AI, GoEmotions) + cliniques → bruit.
Un commentaire Reddit "peur d'un film d'horreur" est étiqueté `détresse=1`.

**Deux changements principaux**

1. **Données cliniques uniquement** :
   - ✅ Kaggle Reddit Depression/SuicideWatch (communautés cliniques)
   - ✅ eRisk25 si disponible (labels cliniques validés CLEF 2025)
   - ❌ DAIR-AI/emotion, GoEmotions (trop bruités)

2. **Mental-BERT comme base** :
   - `mental/mental-bert-base-uncased` : pré-entraîné sur 1.3B tokens santé mentale
   - +3-5% F1 sur tâches cliniques vs DistilBERT standard (Ji et al. NAACL 2022)

### Objectifs

| Métrique | v2 DistilBERT | Cible v3 Mental-BERT |
|----------|:-------------:|:--------------------:|
| Accuracy | 89.0% | ≥ 89% |
| F1 Macro | 86.5% | ≥ 87% |
| **Sensitivité** | ~85% | **≥ 90%** |
| AUC-ROC | N/A | ≥ 0.92 |
| Brier score | N/A | ≤ 0.10 |

In [ ]:
# Étape 1 : installation
# Kaggle a déjà transformers/torch pré-installés — on met à jour pour avoir la dernière version
!pip install -q -U transformers accelerate datasets scikit-learn

In [ ]:
# Étape 2 : vérification GPU + reproductibilité
import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device :', torch.cuda.get_device_name(0))
    print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  Pas de GPU — activer via Settings > Accelerator > GPU T4 x2')

import transformers
print('Transformers :', transformers.__version__)

In [ ]:
# Étape 3 : chemins Kaggle
# ─────────────────────────────────────────────────────────────────────────────
# Sur Kaggle : les datasets ajoutés sont montés dans /kaggle/input/<dataset-slug>/
# Les fichiers de sortie vont dans /kaggle/working/ (téléchargeables depuis l'UI)
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR   = '/kaggle/working/mental_bert_v3'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Chemins des datasets ─────────────────────────────────────────────────────
# Adapter le slug si le nom du dataset Kaggle est différent

# Option A : dataset Reddit Depression (bhavikjikadara/mental-health-dataset)
KAGGLE_PATH_A = '/kaggle/input/mental-health-dataset/Combined Data.csv'

# Option B : Suicide Ideation dataset (nikhileswarkomati/suicide-ideation-dataset)
KAGGLE_PATH_B = '/kaggle/input/suicide-ideation-dataset/Suicide_Detection.csv'

# Option C : eRisk25 (dataset privé Kaggle, si disponible)
ERISK25_PATH = '/kaggle/input/erisk25/final-eriskt2-dataset-with-ground-truth/all_combined'

print('Output dir   :', OUTPUT_DIR)
print()
print('Sources disponibles :')
for label, path in [('Reddit/MH (A)', KAGGLE_PATH_A), ('Suicide (B)', KAGGLE_PATH_B), ('eRisk25 (C)', ERISK25_PATH)]:
    status = '✅' if os.path.exists(path) else '❌ absent'
    print(f'  {label:15s}: {status}  ({path})')

In [ ]:
# Étape 4 : chargement des données (cliniques uniquement)
import re
import json
import glob
import pandas as pd
from sklearn.model_selection import train_test_split


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r"'", '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()


def load_mh_dataset_a(path: str, max_samples: int = 80_000) -> pd.DataFrame:
    """Mental Health Dataset (Kaggle bhavikjikadara) — statut normal/dépression/anxiété..."""
    df = pd.read_csv(path, low_memory=False)
    print('Colonnes A :', list(df.columns))

    text_col   = next((c for c in ['statement', 'text', 'post', 'content'] if c in df.columns), None)
    status_col = next((c for c in ['status', 'label', 'category'] if c in df.columns), None)

    if not text_col or not status_col:
        raise ValueError(f'Colonnes text/status introuvables. Colonnes : {list(df.columns)}')

    df = df[[text_col, status_col]].rename(columns={text_col: 'text', status_col: 'status'})
    df = df.dropna()

    DISTRESS_STATUSES = {'depression', 'suicidal', 'anxiety', 'bipolar disorder', 'ptsd',
                         'depression ', 'suicidal ideation', 'severe depression'}
    NORMAL_STATUSES   = {'normal', 'none', 'no depression', 'healthy'}

    df['label_str'] = df['status'].str.lower().str.strip()
    df['label'] = df['label_str'].apply(
        lambda s: 1 if s in DISTRESS_STATUSES else (0 if s in NORMAL_STATUSES else None)
    )
    df = df[df['label'].notna()].copy()
    df['label'] = df['label'].astype(int)
    df = df[df['text'].str.len() > 20]

    d1 = df[df['label'] == 1]
    n_neg = min(int(len(d1) * 1.5), len(df[df['label'] == 0]))
    d0 = df[df['label'] == 0].sample(n=n_neg, random_state=SEED)
    df = pd.concat([d1, d0], ignore_index=True)

    if max_samples and len(df) > max_samples:
        df, _ = train_test_split(df, train_size=max_samples, stratify=df['label'], random_state=SEED)

    df['text'] = df['text'].apply(clean_text)
    df = df[df['text'].str.len() > 10][['text', 'label']]
    print(f'Dataset A : {len(df)} lignes | {df["label"].value_counts().to_dict()}')
    return df.reset_index(drop=True)


def load_suicide_dataset_b(path: str, max_samples: int = 40_000) -> pd.DataFrame:
    """Suicide Ideation dataset (nikhileswarkomati) — text + class (suicide/non-suicide)."""
    df = pd.read_csv(path, low_memory=False)
    print('Colonnes B :', list(df.columns))

    text_col  = next((c for c in ['text', 'post', 'content'] if c in df.columns), None)
    class_col = next((c for c in ['class', 'label', 'target'] if c in df.columns), None)

    if not text_col or not class_col:
        raise ValueError(f'Colonnes introuvables. Colonnes : {list(df.columns)}')

    df = df[[text_col, class_col]].rename(columns={text_col: 'text', class_col: 'class_label'})
    df = df.dropna()
    df['label'] = (df['class_label'].str.lower().str.strip() == 'suicide').astype(int)
    df = df[df['text'].str.len() > 20]

    if max_samples and len(df) > max_samples:
        df, _ = train_test_split(df, train_size=max_samples, stratify=df['label'], random_state=SEED)

    df['text'] = df['text'].apply(clean_text)
    df = df[df['text'].str.len() > 10][['text', 'label']]
    print(f'Dataset B : {len(df)} lignes | {df["label"].value_counts().to_dict()}')
    return df.reset_index(drop=True)


def load_erisk25(json_dir: str) -> pd.DataFrame:
    """eRisk25 — labels cliniques dépression validés (CLEF 2025).
    
    Fix : OSError (errno 107 — filesystem FUSE déconnecté) attrapé au niveau du
    with open(), pas seulement à l'intérieur. Fichiers illisibles ignorés.
    """
    files = glob.glob(f'{json_dir}/subject_*.json')
    if not files:
        raise FileNotFoundError(f'Aucun fichier subject_*.json dans {json_dir}')

    rows = []
    skipped = 0
    for filepath in files:
        # OSError doit être attrapé autour du with open(), pas seulement inside
        try:
            with open(filepath, encoding='utf-8') as f:
                posts = json.load(f)
        except (OSError, json.JSONDecodeError):
            skipped += 1
            continue

        for post in posts:
            sub    = post.get('submission', {})
            target = sub.get('target')
            if target is None:
                continue
            text = ((sub.get('title') or '') + ' ' + (sub.get('body') or '')).strip()
            if len(text) > 20:
                rows.append({'text': text, 'label': int(bool(target))})

    if skipped:
        print(f'⚠️  eRisk25 : {skipped} fichiers ignorés (FUSE déconnecté ou JSON invalide)')

    df = pd.DataFrame(rows)
    df['text'] = df['text'].apply(clean_text)
    df = df[df['text'].str.len() > 10].drop_duplicates(subset=['text'])
    print(f'eRisk25 : {len(df)} lignes | {df["label"].value_counts().to_dict()}')
    return df.reset_index(drop=True)


# ─── Assembler les sources disponibles ──────────────────────────────────────
frames = []

if os.path.exists(KAGGLE_PATH_A):
    frames.append(load_mh_dataset_a(KAGGLE_PATH_A, max_samples=80_000))
else:
    print(f'⚠️  Dataset A absent : {KAGGLE_PATH_A}')

if os.path.exists(KAGGLE_PATH_B):
    frames.append(load_suicide_dataset_b(KAGGLE_PATH_B, max_samples=40_000))
else:
    print(f'⚠️  Dataset B absent : {KAGGLE_PATH_B}')

if os.path.exists(ERISK25_PATH):
    frames.append(load_erisk25(ERISK25_PATH))
else:
    print(f'ℹ️  eRisk25 absent (optionnel) : {ERISK25_PATH}')

if not frames:
    raise RuntimeError(
        'Aucune source de données trouvée.\n'
        'Ajouter un dataset via Add Data dans le sidebar Kaggle.'
    )

df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['text'])
print(f'\nDataset clinique final : {len(df)} lignes')
print(f'Distribution :\n{df["label"].value_counts()}')

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=SEED)
print(f'\nTrain: {len(train_df)} | Test: {len(test_df)}')

In [ ]:
# Étape 5 : tokenisation avec Mental-BERT
#
# mental/mental-bert-base-uncased :
#   - Pré-entraîné sur 1.3B tokens santé mentale
#     (Reddit : r/depression, r/anxiety, r/SuicideWatch + articles PubMed psychiatrie)
#   - Architecture identique à bert-base-uncased → même pipeline
#   - Publié : Ji et al. 2022 "MentalBERT" — NAACL 2022

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

MODEL_NAME = 'mental/mental-bert-base-uncased'
# MODEL_NAME = 'distilbert-base-uncased'  # fallback si quota HF dépassé

print(f'Chargement tokenizer : {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_ds = Dataset.from_pandas(train_df[['text', 'label']]).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df[['text', 'label']]).map(tokenize, batched=True)

train_ds = train_ds.rename_column('label', 'labels')
test_ds  = test_ds.rename_column('label', 'labels')
train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print('Tokenisation OK')
print('Exemple token ids :', train_ds[0]['input_ids'][:10])

In [ ]:
# Étape 6 : entraînement
#
# Changements v3 vs v2 :
#   - metric_for_best_model = 'recall_1' (sensitivité) au lieu de 'f1'
#   - weight_decay = 0.01 → régularisation L2
#   - warmup_ratio = 0.1 → learning rate progressif
#   - EarlyStopping patience=2
#
# Note Kaggle T4 x2 : activer "Use two GPUs" dans Settings pour accélérer

from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    proba_pos = logits[:, 1]
    try:
        auc = roc_auc_score(labels, proba_pos)
    except Exception:
        auc = 0.0
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
        'recall_1': recall_score(labels, preds, pos_label=1),   # ← sensitivité
        'recall_0': recall_score(labels, preds, pos_label=0),   # ← spécificité
        'auc_roc':  auc,
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='recall_1',
    greater_is_better=True,
    logging_steps=100,
    report_to='none',
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f'Début entraînement : {MODEL_NAME}')
print(f'Train : {len(train_ds)} | Test : {len(test_ds)}')
trainer.train()

In [ ]:
# Étape 7 : évaluation clinique complète
from sklearn.metrics import classification_report, brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

predictions = trainer.predict(test_ds)
logits      = predictions.predictions
labels      = predictions.label_ids
preds       = np.argmax(logits, axis=-1)

# Softmax → probabilités
exp_logits    = np.exp(logits - logits.max(axis=1, keepdims=True))
proba_pos     = exp_logits[:, 1] / exp_logits.sum(axis=1)

sensitivity = recall_score(labels, preds, pos_label=1)
specificity = recall_score(labels, preds, pos_label=0)
auc         = roc_auc_score(labels, proba_pos)
brier       = brier_score_loss(labels, proba_pos)
f1_macro    = f1_score(labels, preds, average='macro')
acc         = accuracy_score(labels, preds)

print('=' * 60)
print(f'  RÉSULTATS v3 — {MODEL_NAME}')
print('  ' + '-' * 56)
print(f'  Accuracy       : {acc:.4f}')
print(f'  F1 Macro       : {f1_macro:.4f}')
print(f'  Sensitivité    : {sensitivity:.4f}  ← objectif ≥ 0.90')
print(f'  Spécificité    : {specificity:.4f}')
print(f'  AUC-ROC        : {auc:.4f}          ← objectif ≥ 0.92')
print(f'  Brier score    : {brier:.4f}          ← calibration (0=parfait)')
print('=' * 60)
print()
print(classification_report(labels, preds, target_names=['non-détresse', 'détresse']))

# Courbe de calibration
prob_true, prob_pred = calibration_curve(labels, proba_pos, n_bins=10)
plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], 'k--', label='Calibration parfaite')
plt.plot(prob_pred, prob_true, 's-', label=MODEL_NAME)
plt.xlabel('Score prédit'); plt.ylabel('Fraction vrais positifs')
plt.title('Calibration des probabilités — v3')
plt.legend(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/calibration_curve.png', dpi=120)
plt.show()

In [ ]:
# Étape 8 : sauvegarde dans /kaggle/working/
# Télécharger ensuite via l'onglet "Output" de Kaggle
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Modèle sauvegardé → {OUTPUT_DIR}')
print('Fichiers :', os.listdir(OUTPUT_DIR))

summary = {
    'model_name':   MODEL_NAME,
    'platform':     'Kaggle',
    'accuracy':     round(float(acc), 4),
    'f1_macro':     round(float(f1_macro), 4),
    'recall_1':     round(float(sensitivity), 4),
    'recall_0':     round(float(specificity), 4),
    'auc_roc':      round(float(auc), 4),
    'brier_score':  round(float(brier), 4),
    'dataset_size': len(df),
    'sources':      'Kaggle MH Dataset + Suicide Ideation + eRisk25 (clinical only)',
    'targets_met':  {
        'sensitivity_gte_090': float(sensitivity) >= 0.90,
        'auc_roc_gte_092':     float(auc) >= 0.92,
        'f1_macro_gte_085':    float(f1_macro) >= 0.85,
    },
}
with open(f'{OUTPUT_DIR}/eval_metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)

print()
print(json.dumps(summary, indent=2))
print()
print('─' * 60)
targets = summary['targets_met']
print('Objectifs atteints :')
print(f'  Sensitivité ≥ 0.90 : {"✅" if targets["sensitivity_gte_090"] else "❌"}  ({sensitivity:.4f})')
print(f'  AUC-ROC ≥ 0.92     : {"✅" if targets["auc_roc_gte_092"] else "❌"}  ({auc:.4f})')
print(f'  F1 Macro ≥ 0.85    : {"✅" if targets["f1_macro_gte_085"] else "❌"}  ({f1_macro:.4f})')
print('─' * 60)
print(f'\n→ Télécharger le dossier mental_bert_v3/ depuis l\'onglet Output')

## Étapes suivantes

### 1. Récupérer le modèle entraîné
- Kaggle > votre notebook > onglet **Output** > télécharger `mental_bert_v3/`
- Placer dans `models/fine_tuned/mental_bert_v3/` du projet

### 2. Datasets recommandés (Add Data dans Kaggle)

| Dataset | Slug | Contenu |
|---|---|---|
| Mental Health Dataset | `bhavikjikadara/mental-health-dataset` | 53K posts, 7 catégories |
| Suicide Ideation | `nikhileswarkomati/suicide-ideation-dataset` | 232K Reddit posts |

### 3. Intégration production

L'API accepte `VITE_MODEL_TYPE=distilbert` (env var Vercel).
Pour activer Mental-BERT v3 en prod, il faudrait :
- Uploader le modèle sur HuggingFace Hub (repo privé ou public)
- Ou inclure dans le Docker image (attention : ~440MB supplémentaires)
- Adapter `MODEL_TYPE=mental_bert_v3` dans la config API

### 4. Phase v4 — Multi-label clinique

Prédire directement `[depression, anxiety, burnout]` depuis le texte
pour remplacer la détection heuristique par mots-clés dans le navigateur.

Données nécessaires :
- Depression : eRisk25 ✅
- Anxiety : CLEF eRisk Task 1 ou Kaggle anxiety dataset
- Burnout : SMHD si accès obtenu